# Warp on the NPE emulator — validated on the DB examples

HW-faithful warp emulation following `warp_m/HW_WARP.md` and the final format
decisions — all operands are **plain INTs on an MX container with a zero
exponent field** ("NOT MX INT BUT INT"):

| tensor | format | content |
|---|---|---|
| reference | **uint8** (round+clamp, NOT quantized to MX) | values 0..255, `mx-e4bm8`, exp = 0 |
| grid | **int16** `q = round(coord·2^fr)` per axis | **absolute** sample coordinate in pixels, `mx-e4bm16`, exp = 0 |
| TRC output | int in `[0, 2^(frx+fry)]` | corner indices + `w00..w11` (Σ = `2^(frx+fry)`), `mx-e4bm16`, exp = 0 |
| **output** | **int24 (relux)** | PCU: `acc >> (frx+fry)` round-half-up, 24-bit relux (int8 clamp only if you ask for it) |

Pipe: **TRC** (`trc_warp`: msb/lsb split `idx = q>>fr`, `frac = q & (2^fr−1)`;
corner addresses clipped to `[0, dim−2]` + the 4 weights) → **TRU-A**
(`tru_a_warp`: 2-line corner fetch) → **BCU** (4-term MAC, exponents all zero)
→ **PCU** (`build_warp_c_struct_arr`).

Why `mx-e4bm16` for the grid/weights: the TRC weight is an exact integer over
`2^(frx+fry)` (up to `2^14`), which does not fit the 8-bit mantissa lane.

Fraction bits follow the reference notebook (`EL_nb/new_warp_torch.ipynb`,
`grid_quantization`), per axis:

```
fr = grid_bit_width − floor(log2(ceil(max|coord_axis|))) − 2      (grid_bit_width = 16)
```

capped so `frx + fry ≤ 14` (weight lane), reducing both axes evenly on small
frames. No `auto_warp_fl` anymore — the packer derives it from the data.

Validated against `/home/eliyahul/ai_ws/MX_FORMAT/nndof_thetis_MX/warp_args`
(14 examples). Their exact float conventions were recovered first:

- 13 examples: **normalized-acT deltas, align_corners=True** ->
  `delta_px = flow · [(W−1)/2, (H−1)/2]`
- example 2: **pixel deltas** directly.
- channel 0 = x, channel 1 = y; same-size warp (out shape == in shape).

Assertions per example: (a) **exact integer equality** vs an integer golden
mirroring TRC → BCU → PCU on the same packed operands; (b) the packer follows
the notebook's `grid_quantization` bit-for-bit; (c) mse vs the stored float
output is reported next to its reference-rounding floor.

In [ ]:
import os
import sys
_here = os.getcwd()
_cands = [_here, os.path.dirname(_here)]  # EL_nb/ -> parent
for _c in _cands:
    if os.path.isdir(os.path.join(_c, 'npe_mx_emu')) and _c not in sys.path:
        sys.path.insert(0, _c)
        break

import numpy as np
import torch

from npe_mx_emu import (
    CalibratedMXWarp, LayerMXQParams,
    build_warp_c_struct_arr, pack_warp_grid, pack_warp_reference,
)

DB = "/home/eliyahul/ai_ws/MX_FORMAT/nndof_thetis_MX/warp_args"
N_EX = 14
GRID_BITS = 16

# DB conventions (recovered): normalized-acT deltas for all examples except ex2
AC_T = [i for i in range(N_EX) if i != 2]


def load_example(i):
    """(ref, flow, out_db) float64 numpy from the stored DB."""
    ref = torch.load(f"{DB}/tenInputs/tenInput{i}.pt").double().numpy()   # (B,C,H,W)
    flow = torch.load(f"{DB}/tenFlows/tenFlow{i}.pt").double().numpy()    # (B,2,H,W)
    out_db = torch.load(f"{DB}/outputs/output{i}.pt").double().numpy()    # (B,C,H,W)
    return ref, flow, out_db


def acT_deltas(flow, W, H):
    scT = np.array([[(W - 1) / 2, (H - 1) / 2]]).reshape(1, 2, 1, 1)
    return flow * scT


def absolute_grid(flow, i, W, H):
    """Absolute sample coordinates [B, 2, H, W] in pixels (channel 0 = x, 1 = y)."""
    deltas = acT_deltas(flow, W, H) if i in AC_T else flow
    loc = np.dstack(np.meshgrid(np.arange(W), np.arange(H)))[None].transpose(0, 3, 1, 2)
    return loc + deltas

## Integer golden (numpy, 1:1 with the emulator)

Reference = `round` to int (uint8, zero exponent). Grid = int16
`q = round(coord·2^fr)` with **per-axis** `fr` from the rule above. TRC =
shift/mask split + corner clip with the fraction re-derived after the clip
(replicate semantics, like `np_warp`). BCU = 4-term integer MAC.
PCU = `acc >> (frx+fry)` **round-half-up**, then 24-bit relux (no int8 clamp).

The emulator must reproduce this **exactly**, integer for integer.

In [ ]:
def quantize_reference(ref_np):
    """uint8: round + clamp, zero exponent (NOT MX-quantized)."""
    return np.round(ref_np).clip(0, 255).astype(np.int64)


def golden_int(ref_i, q, frx, fry):
    """Integer golden, 1:1 with TRC -> TRU-A -> BCU -> PCU."""
    C, H, W = ref_i.shape
    Sx, Sy = 1 << frx, 1 << fry
    r = np.moveaxis(ref_i, 0, -1)                                 # (H, W, C)

    idx_x, f_x = q[0] >> frx, q[0] & (Sx - 1)                     # TRC msb/lsb split
    idx_y, f_y = q[1] >> fry, q[1] & (Sy - 1)
    ix = np.clip(idx_x, 0, W - 2)                                 # corner addresses
    iy = np.clip(idx_y, 0, H - 2)

    wx1 = np.clip((idx_x - ix) * Sx + f_x, 0, Sx)                 # replicate fraction
    wy1 = np.clip((idx_y - iy) * Sy + f_y, 0, Sy)
    wx0, wy0 = Sx - wx1, Sy - wy1

    acc = (r[iy,     ix]     * (wx0 * wy0)[..., None]             # BCU: 4-term MAC
         + r[iy,     ix + 1] * (wx1 * wy0)[..., None]
         + r[iy + 1, ix]     * (wx0 * wy1)[..., None]
         + r[iy + 1, ix + 1] * (wx1 * wy1)[..., None])            # (Ho, Wo, C)

    k = frx + fry                                                 # PCU: round + dequant
    x_floor = acc >> k
    rem = acc - (x_floor << k)
    out = x_floor + (rem >= (1 << (k - 1)))                       # round half up
    return np.moveaxis(out, -1, 0).astype(np.int64)               # 24-bit relux: no clamp

## Run the 14 examples through `CalibratedMXWarp`

Per example: convert flow -> pixel deltas -> **absolute** coordinates, pack
with `pack_warp_grid` (int16 grid + per-axis `fr`), run the layer, compare
**exact integers** vs the golden, and report mse / max|err| vs the stored
float output.

The golden uses the `q`/`fr` produced by the packer, so the check that the
packer itself follows the notebook rule is a separate assertion (numpy
`round(coord·2^fr)` equality).

In [ ]:
qp = LayerMXQParams(x_cal=0, w_cal=0, out_cal=0)

rows = []
for i in range(N_EX):
    ref, flow, out_db = load_example(i)                       # (B,C,H,W), (B,2,H,W)
    B, C, H, W = ref.shape
    coords = absolute_grid(flow, i, W, H)                     # absolute pixels (x, y)
    coords_t = torch.tensor(coords, dtype=torch.float64)

    # ---- emulator: float reference + absolute coords in, integer out ----
    warp = CalibratedMXWarp(channels=C, qparams=qp, grid_bit_width=GRID_BITS)
    out = warp(torch.tensor(ref, dtype=torch.float64), coords_t).long().numpy()

    # ---- packed operands (also what the pipe sees internally) ----
    mx_grid, fr = pack_warp_grid(coords_t, grid_bit_width=GRID_BITS)
    frx, fry = fr.numpy().tolist()
    q = mx_grid.data.numpy()

    # packer == notebook grid_quantization (per-axis round(coord * 2**fr))
    q_ref = np.round(coords * np.array([2.0 ** frx, 2.0 ** fry]).reshape(1, 2, 1, 1))
    assert np.array_equal(q, q_ref.astype(np.int64)), "pack_warp_grid != notebook grid_quantization"

    # uncapped per-axis rule, for the printout (the cap only kicks in on small frames)
    fr_raw = GRID_BITS - np.floor(np.log2(np.maximum(np.ceil(np.abs(coords).max(axis=(0, 2, 3))), 1))) - 2

    # ---- integer golden must match exactly ----
    ref_i = quantize_reference(ref)
    gold = np.stack([golden_int(ref_i[b], q[b], frx, fry) for b in range(B)])
    exact = np.array_equal(out, gold)

    mse = ((out.astype(np.float64) - out_db) ** 2).mean()
    mad = np.abs(out.astype(np.float64) - out_db).max()
    rr = ((np.round(ref) - ref) ** 2).mean()
    rows.append((i, tuple(ref.shape), (frx, fry), exact, mse, mad, rr))
    print(f"ex{i:2d} {str(tuple(ref.shape)):14s} fr={frx},{fry} (raw {fr_raw.astype(int)}) "
          f"exact-int={exact}  mse vs DB={mse:.4f}  max|err|={mad:.4f}  (ref-round floor {rr:.4f})")

assert all(r[3] for r in rows), "integer mismatch between emulator and golden"
print()
print(f"mse vs DB outputs: [{min(r[4] for r in rows):.4f} .. {max(r[4] for r in rows):.4f}]")
print(f"all max|err| <= {max(r[5] for r in rows):.4f}")
print("integer golden: exact on all examples")

### Why the mse is ~0.02–0.05 and not ~1e-12

The stored DB outputs are **float** warps of **float** inputs. The integer
emulator's error vs them decomposes into:

1. **reference rounding** (`round(input)`): mean-square ≈ `1/12` per element —
   the `ref-round floor` column; this dominates;
2. **PCU integer rounding** of the warped value (same order);
3. grid fraction quantization (`2^-fr` px, `fr` printed per example): the
   coordinate error, visible only on high-gradient edges.

Everything is quantization error of the **formats** — the warp arithmetic
itself is exact (asserted above, integer for integer).

## `CalibratedMXWarp` interface

`forward(reference, coords)` is the whole pipeline: it packs the reference
(uint8, zero exp) and the grid (int16, `mx-e4bm16`, zero exp), recalibrates
the PCU c_structs for the data-dependent `fr`, and returns the raw integer
output (the warped value, since `out_cal = 0`; the float value is
`out * 2**-output_scale_log2`).

The bypass below feeds `MXWarp_Loops` the packed operands directly — it must
equal both the wrapper and the golden.

In [ ]:
i = 0
ref, flow, out_db = load_example(i)
B, C, H, W = ref.shape
coords = absolute_grid(flow, i, W, H)

qp = LayerMXQParams(x_cal=0, w_cal=0, out_cal=0)

# ---- wrapper: float reference + absolute coords ----
w = CalibratedMXWarp(channels=C, qparams=qp, grid_bit_width=GRID_BITS)
out_wrapper = w(torch.tensor(ref, dtype=torch.float64),
                torch.tensor(coords, dtype=torch.float64)).long().numpy()

# ---- bypass: hand the packed operands to the pipe stage directly ----
mx_ref = pack_warp_reference(torch.tensor(ref, dtype=torch.float64))
mx_grid, fr = pack_warp_grid(torch.tensor(coords, dtype=torch.float64), grid_bit_width=GRID_BITS)
out_bypass = w.mx_warp(mx_ref, mx_grid, fr).long().numpy()

ref_i = quantize_reference(ref)
q = mx_grid.data.numpy()
frx, fry = fr.numpy().tolist()
gold = np.stack([golden_int(ref_i[b], q[b], frx, fry) for b in range(B)])

print(f"example {i}: fr={frx},{fry}  calibrated shift={w.mx_warp.c_struct_arr[0].shift} "
      f"(+ ref e_min_layer={int(mx_ref.e_min_layer)} => effective {w.mx_warp.c_struct_arr[0].shift + int(mx_ref.e_min_layer)})")
print("wrapper == golden:", np.array_equal(out_wrapper, gold))
print("bypass  == golden:", np.array_equal(out_bypass, gold))
assert np.array_equal(out_bypass, gold)
assert np.array_equal(out_wrapper, gold)

## Visualization — example 2 (pixel-delta flow)

Channel 3 of batch 0: input, stored DB output, emulator output, and the
|diff| map (int-format quantization).

In [ ]:
import matplotlib.pyplot as plt

i = 2
ref, flow, out_db = load_example(i)
B, C, H, W = ref.shape
coords = absolute_grid(flow, i, W, H)

qp = LayerMXQParams(x_cal=0, w_cal=0, out_cal=0)
warp = CalibratedMXWarp(channels=C, qparams=qp, grid_bit_width=GRID_BITS)
out = warp(torch.tensor(ref, dtype=torch.float64),
           torch.tensor(coords, dtype=torch.float64)).long().numpy()
frx, fry = warp._frac_key

b, c = 0, 3
fig, axs = plt.subplots(1, 4, figsize=(16, 4))
axs[0].imshow(ref[b, c], cmap='viridis');    axs[0].set_title(f'in (ch{c})')
axs[1].imshow(out_db[b, c], cmap='viridis'); axs[1].set_title('DB output')
axs[2].imshow(out[b, c], cmap='viridis');    axs[2].set_title('emulator (int24 relux)')
d = axs[3].imshow(np.abs(out[b, c].astype(np.float64) - out_db[b, c]), cmap='hot')
axs[3].set_title('|emulator - DB|')
fig.colorbar(d, ax=axs[3])
fig.suptitle(f'warp DB example {i}  (fr={frx},{fry})')
plt.tight_layout(); plt.show()